# 1. Install Modules

In [ ]:
# Bioinformatics Tools (Ubuntu)
!sudo apt-get update
!sudo apt-get install -y fastp flash bwa samtools

# Python Library
!pip3 install biopython cutadapt pysam --break-system-packages

# 2 Trimming and Discard trimmed sample

In [ ]:
import subprocess
import glob
import os

# Specify the folder containing your input files.
# Specify the folder where you want to save the untrimmed (adapter-free) sequences.
input_folder = "fastq"
untrimmed_output_folder = "fastq/stage2/A_untrimmed_output"

# Define the adapter sequences for R1 and R2.
adapter_sequence_r1 = "AGATCGGAAGAGCACACGTCTGAACTCCAGTCAC"
adapter_sequence_r2 = "AGATCGGAAGAGCGTCGTGTAGGGAAAGAGTGT"

# Use glob to get a list of all input file pairs (R1 and R2) in the folder.
input_file_pairs = []
for input_r1 in glob.glob(os.path.join(input_folder, "*_R1.fastq.gz")):
    # Assuming R2 files have the same naming format as R1 files.
    input_r2 = input_r1.replace("_R1.fastq.gz", "_R2.fastq.gz")
    if os.path.exists(input_r2):  # Ensure R2 file exists.
        input_file_pairs.append({"r1": input_r1, "r2": input_r2})

# Create the output folder if it doesn't exist.
os.makedirs(untrimmed_output_folder, exist_ok=True)

for input_files in input_file_pairs:
    input_r1 = input_files["r1"]
    input_r2 = input_files["r2"]

    # Define output file paths for untrimmed (clean, adapter-free) sequences.
    untrimmed_r1 = os.path.join(untrimmed_output_folder, os.path.basename(input_r1).replace(".fastq.gz", "_untrimmed.fastq.gz"))
    untrimmed_r2 = os.path.join(untrimmed_output_folder, os.path.basename(input_r2).replace(".fastq.gz", "_untrimmed.fastq.gz"))

    # Use cutadapt to keep only untrimmed sequences (completely adapter-free).
    result = subprocess.run([
        "cutadapt",
        "-a", adapter_sequence_r1,  # Adapter for R1
        "-A", adapter_sequence_r2,  # Adapter for R2
        "-O", "15",                  # Minimum overlap for adapter trimming
        "--discard-trimmed",         # Discard sequences where trimming occurred
        "-o", untrimmed_r1,          # Save only untrimmed R1 reads
        "-p", untrimmed_r2,          # Save only untrimmed R2 reads
        input_r1, input_r2
    ], capture_output=True, text=True)

    # Log the result.
    if result.returncode == 0:
        print(f"Untrimmed sequences saved: {untrimmed_r1}, {untrimmed_r2}")
    else:
        print(f"Error processing {input_r1} and {input_r2}:\n{result.stderr}")

# 3. Q filtering

In [ ]:
import os
import subprocess

# Quality threshold (Phred score)
quality_threshold = 30

# Set input and output folders
input_folder = "fastq/stage2/A_untrimmed_output"
output_folder = "fastq/stage2/B_Qfiltered"

# Create the output folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# Iterate through files in the input folder, processing only those ending with "_untrimmed.fastq.gz"
for filename in os.listdir(input_folder):
    if filename.endswith("_untrimmed.fastq.gz"):
        # Input file path
        input_file = os.path.join(input_folder, filename)
        
        # Output filename (e.g., sample_untrimmed.fastq.gz -> sample_Qfiltered.fastq.gz)
        output_file = os.path.join(
            output_folder, 
            filename.replace("_untrimmed.fastq.gz", "_Qfiltered.fastq.gz")
        )
        
        # Execute fastp in single-end mode for each file
        subprocess.call([
            "fastp",
            "-i", input_file,                      # Input file
            "-o", output_file,                     # Output file
            "-q", str(quality_threshold),          # Quality threshold for a base to be qualified
            "-u", "15",                            # Discard reads if the percentage of unqualified bases is >= 15%
            "-l", "151",                           # Minimum read length to keep
            "--cut_mean_quality", "30",            # Discard reads if mean quality is less than 30
            "--html", f"{output_file}.html",       # HTML report file path
            "--json", f"{output_file}.json"        # JSON report file path
        ])
        
        print(f"Filtering for {filename} is complete.\n"
              f"Output FASTQ : {output_file}\n"
              f"Reports      : {output_file}.html / {output_file}.json\n")

print("All filtering processes are done.")

# 4. Match Paired-End Read IDs

In [ ]:
import gzip
import glob
import os

def extract_matching_reads(r1_path, r2_path, out_r1_path, out_r2_path):
    def get_read_id(header):
        # Extract ID from the FASTQ header
        return header.split()[0].replace('/1', '').replace('/2', '')

    r1_ids = set()
    r2_ids = set()

    # Extract all read IDs from the R1 file
    with gzip.open(r1_path, 'rt') as r1_file:
        while True:
            header = r1_file.readline()
            if not header:
                break
            r1_ids.add(get_read_id(header.strip()))
            # Skip the other 3 lines of the read (sequence, +, quality)
            [r1_file.readline() for _ in range(3)] 

    # Extract all read IDs from the R2 file
    with gzip.open(r2_path, 'rt') as r2_file:
        while True:
            header = r2_file.readline()
            if not header:
                break
            r2_ids.add(get_read_id(header.strip()))
            [r2_file.readline() for _ in range(3)]

    # Find common and unique IDs
    matching_ids = r1_ids & r2_ids
    r1_only = r1_ids - r2_ids
    r2_only = r2_ids - r1_ids

    print(f"Processing {os.path.basename(r1_path)} and {os.path.basename(r2_path)}")
    print(f"Total R1 IDs: {len(r1_ids)}, Total R2 IDs: {len(r2_ids)}, Matching IDs: {len(matching_ids)}")
    print(f"IDs only in R1: {len(r1_only)}, IDs only in R2: {len(r2_only)}\n")

    # Create the output directory if it doesn't exist
    os.makedirs(os.path.dirname(out_r1_path), exist_ok=True)

    # Function to write only the reads with matching IDs to a new file
    def write_matching_reads(input_path, output_path, matching_ids):
        with gzip.open(input_path, 'rt') as infile, gzip.open(output_path, 'wt') as outfile:
            while True:
                lines = [infile.readline() for _ in range(4)]
                if not lines[0]:
                    break
                read_id = get_read_id(lines[0].strip())
                if read_id in matching_ids:
                    outfile.writelines(lines)

    # Write the filtered R1 and R2 files
    write_matching_reads(r1_path, out_r1_path, matching_ids)
    write_matching_reads(r2_path, out_r2_path, matching_ids)

# --------------------------
# Apply to all file pairs
# --------------------------

input_folder = "fastq/stage2/B_Qfiltered"
output_folder = "fastq/stage2/C_id_matched"

# Find all R1 files
r1_files = glob.glob(os.path.join(input_folder, "*_R1_Qfiltered.fastq.gz"))

# For each R1, find the corresponding R2 file and run the process
for r1_file in r1_files:
    r2_file = r1_file.replace("_R1_Qfiltered.fastq.gz", "_R2_Qfiltered.fastq.gz")
    
    if os.path.exists(r2_file):
        # Set the output file paths
        base_name = os.path.basename(r1_file).replace("_R1_Qfiltered.fastq.gz", "")
        out_r1 = os.path.join(output_folder, f"{base_name}_ID_match_R1.fastq.gz")
        out_r2 = os.path.join(output_folder, f"{base_name}_ID_match_R2.fastq.gz")
        
        # Execute the function
        extract_matching_reads(r1_file, r2_file, out_r1, out_r2)
    else:
        print(f"Warning: Corresponding R2 file not found for {r1_file}. Skipping.")

# 5 Merge W/ Flash

## 5.1 R1(Front, Back), R2(Front, Back) Fragmentation

In [ ]:
import gzip
import glob
import os

def split_fastq_by_position(r1_path, r2_path, n, output_dir):
    """Splits each read in R1 and R2 files into front and back parts."""
    os.makedirs(output_dir, exist_ok=True)

    sample_base = os.path.basename(r1_path).replace("_ID_match_R1.fastq.gz", "")
    r1_f_path = os.path.join(output_dir, f"{sample_base}_R1_F.fastq.gz")
    r1_b_path = os.path.join(output_dir, f"{sample_base}_R1_B.fastq.gz")
    r2_f_path = os.path.join(output_dir, f"{sample_base}_R2_F.fastq.gz")
    r2_b_path = os.path.join(output_dir, f"{sample_base}_R2_B.fastq.gz")

    with gzip.open(r1_path, 'rt') as r1_file, \
         gzip.open(r2_path, 'rt') as r2_file, \
         gzip.open(r1_f_path, 'wt') as r1_f_out, \
         gzip.open(r1_b_path, 'wt') as r1_b_out, \
         gzip.open(r2_f_path, 'wt') as r2_f_out, \
         gzip.open(r2_b_path, 'wt') as r2_b_out:

        while True:
            r1_lines = [r1_file.readline() for _ in range(4)]
            r2_lines = [r2_file.readline() for _ in range(4)]

            if not r1_lines[0] or not r2_lines[0]:
                break

            header1, seq1, plus1, qual1 = [line.strip() for line in r1_lines]
            header2, seq2, plus2, qual2 = [line.strip() for line in r2_lines]

            # Split R1 read
            r1_f_out.write(f"{header1}\n{seq1[:151-n]}\n{plus1}\n{qual1[:151-n]}\n")
            r1_b_out.write(f"{header1}\n{seq1[-n:]}\n{plus1}\n{qual1[-n:]}\n")
            # Split R2 read
            r2_f_out.write(f"{header2}\n{seq2[:151-n]}\n{plus2}\n{qual2[:151-n]}\n")
            r2_b_out.write(f"{header2}\n{seq2[-n:]}\n{plus2}\n{qual2[-n:]}\n")

    print(f"✅ Split complete for: {sample_base} → {output_dir} (N={n})")

# -----------------------------------
# Apply the split function to all files
# -----------------------------------

input_folder = "fastq/stage2/C_id_matched"
output_folder = "fastq/stage2/D_split_reads"
os.makedirs(output_folder, exist_ok=True)

# [IMPORTANT] Determine the N-value (merge length) for each sample prefix 
# using the results from 'DNA_data_storage_analysis_stage1.ipynb' before proceeding.
sample_n_mapping = {
    # "0N": 126,
    # "1D": 126,
    # "2S": 126,
    # "3SP": 122,
    # "4G": 126,
    # "5I": 126,
    # "6S": 124,
    # "7T": 120,
}


# Find all R1 files
r1_files = glob.glob(os.path.join(input_folder, "*_ID_match_R1.fastq.gz"))

for r1_file in r1_files:
    r2_file = r1_file.replace("_R1.fastq.gz", "_R2.fastq.gz")

    if not os.path.exists(r2_file):
        print(f"⚠️ Matching R2 file not found: {r2_file}")
        continue

    # Find the corresponding N value based on the filename prefix
    matched_n = None
    for prefix, n_value in sample_n_mapping.items():
        if prefix in os.path.basename(r1_file):
            matched_n = n_value
            break

    if matched_n is None:
        print(f"⚠️ Could not find N value for: {r1_file} → Skipping")
        continue

    # Execute the split function
    split_fastq_by_position(r1_file, r2_file, matched_n, output_folder)

## 5.2 R2 DNA reverse complementary

In [ ]:
import gzip
import glob
import os
from Bio import SeqIO
from Bio.SeqRecord import SeqRecord

def reverse_complement_fastq(input_fastq_path, output_fastq_path):
    # Reads a FASTQ file, creates the reverse complement of each record, and writes it to a new file.
    with gzip.open(input_fastq_path, "rt") as infile, gzip.open(output_fastq_path, "wt") as outfile:
        for record in SeqIO.parse(infile, "fastq"):
            # Create the reverse complement record, preserving the ID and description
            rev_comp_record = record.reverse_complement(id=True, description=True)
            SeqIO.write(rev_comp_record, outfile, "fastq")
            
    print(f"✅ Reverse complemented: {os.path.basename(output_fastq_path)}")

# --------------------------------------------------
# Perform reverse complement on all relevant R2 files
# --------------------------------------------------

input_folder = "fastq/stage2/D_split_reads"
os.makedirs(input_folder, exist_ok=True)

# Find only the R2 front (F) and back (B) fragment files
input_files = glob.glob(os.path.join(input_folder, "*_R2_[BF].fastq.gz"))

for input_path in input_files:
    base = os.path.basename(input_path)
    # Remove the .fastq.gz extension to create a new filename
    name_without_ext = base.replace(".fastq.gz", "")
    output_path = os.path.join(input_folder, f"{name_without_ext}_revcomp.fastq.gz")
    
    reverse_complement_fastq(input_path, output_path)

## 5.3 [R1_back]-[R2_back] merge (FLASH)

In [ ]:
import os
import glob
import subprocess

# === Folder Setup ===
input_folder = "fastq/stage2/D_split_reads"
output_folder = "fastq/stage2/E_merged_output"
os.makedirs(output_folder, exist_ok=True)

# [IMPORTANT] Determine the N-value (merge length) for each sample prefix 
# using the results from 'DNA_data_storage_analysis_stage1.ipynb' before proceeding.
sample_n_mapping = {
    # "0N": 126,
    # "1D": 126,
    # "2S": 126,
    # "3SP": 122,
    # "4G": 126,
    # "5I": 126,
    # "6S": 124,
    # "7T": 120,
}

# === Find List of all R1_B Files ===
r1_files = glob.glob(os.path.join(input_folder, "*_R1_B.fastq.gz"))

print(f"🔎 Found {len(r1_files)} R1_B files.")

# === Process Each R1_B File ===
for r1_path in r1_files:
    sample_base = os.path.basename(r1_path).replace("_R1_B.fastq.gz", "")
    r2_path = os.path.join(input_folder, f"{sample_base}_R2_B.fastq.gz")

    if not os.path.exists(r2_path):
        print(f"⚠️ Matching R2_B file not found for {sample_base} → Skipping.")
        continue

    # Find the corresponding N value for the filename
    matched_n = None
    for prefix, n_value in sample_n_mapping.items():
        if prefix in sample_base:
            matched_n = n_value
            break

    if matched_n is None:
        print(f"⚠️ No N value matched for {sample_base} → Skipping.")
        continue

    output_name = f"{sample_base}_FLASH"

    print(f"🔵 Running FLASH for sample: {sample_base} (N={matched_n})")

    try:
        # Execute the FLASH command
        subprocess.check_call([
            "flash",
            "-m", str(matched_n),   # minimum overlap
            "-M", str(matched_n),   # Maximum overlap
            "-o", output_name,      # Output file prefix
            "-d", output_folder,    # Output directory
            r1_path,
            r2_path
        ])
        print(f"✅ FLASH merging complete → {os.path.join(output_folder, output_name)}.extendedFrags.fastq")
    except subprocess.CalledProcessError as e:
        print(f"❌ FLASH merging failed for {sample_base}: {e}")

## 5.4 Assemble 
## R1_Front - [R1_Back]-[R2_Back]_merged (FLASH) - R2_Front_ReverseComplement

In [ ]:
import os
import gzip
import glob
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

def load_fastq_to_dict(file_path):
    """Loads a FASTQ file into a dictionary: key=read_id, value=(sequence, quality)."""
    data = {}
    open_func = gzip.open if file_path.endswith(".gz") else open

    with open_func(file_path, "rt") as handle:
        for record in SeqIO.parse(handle, "fastq"):
            seq = str(record.seq)
            qual = record.letter_annotations["phred_quality"]
            data[record.id] = (seq, qual)
    return data

def assemble_fastq(r1_path, merged_path, r2_path, output_path):
    """Assembles the final sequence from R1_F, Merged, and R2_F_revcomp fragments."""
    print(f"🔄 Assembling for sample: {os.path.basename(output_path)}")
    r1_dict = load_fastq_to_dict(r1_path)
    r2_dict = load_fastq_to_dict(r2_path)

    with open(merged_path, "r") as merged_file, gzip.open(output_path, "wt") as output_file:
        for record in SeqIO.parse(merged_file, "fastq"):
            read_id = record.id
            merged_seq = str(record.seq)
            merged_qual = record.letter_annotations["phred_quality"]

            # A read must have corresponding R1 and R2 fragments to be assembled.
            if read_id not in r1_dict or read_id not in r2_dict:
                continue  

            r1_seq, r1_qual = r1_dict[read_id]
            r2_seq, r2_qual = r2_dict[read_id]

            # Concatenate in order: R1_F → Merged_Fragment → R2_F_revcomp
            full_seq = r1_seq + merged_seq + r2_seq
            full_qual = r1_qual + merged_qual + r2_qual

            new_record = SeqRecord(
                Seq(full_seq),
                id=read_id,
                description="",
                letter_annotations={"phred_quality": full_qual}
            )

            SeqIO.write(new_record, output_file, "fastq")

    print(f"✅ Assembled FASTQ saved: {output_path}")

# ===== Batch processing =====

# Path setup
input_merged_folder = "fastq/stage2/E_merged_output"
input_split_folder = "fastq/stage2/D_split_reads"
output_folder = "fastq/stage2/1_assemble"
os.makedirs(output_folder, exist_ok=True)

# Get a list of all merged files from FLASH
merged_files = glob.glob(os.path.join(input_merged_folder, "*_FLASH.extendedFrags.fastq"))

print(f"🔍 Found {len(merged_files)} merged samples to assemble.")

for merged_file in merged_files:
    sample_base = os.path.basename(merged_file).replace("_FLASH.extendedFrags.fastq", "")

    r1_path = os.path.join(input_split_folder, f"{sample_base}_R1_F.fastq.gz")
    r2_path = os.path.join(input_split_folder, f"{sample_base}_R2_F_revcomp.fastq.gz")
    output_path = os.path.join(output_folder, f"{sample_base}_assemble.fastq.gz")

    if os.path.exists(r1_path) and os.path.exists(r2_path):
        assemble_fastq(r1_path, merged_file, r2_path, output_path)
    else:
        print(f"⚠️ Missing split files for {sample_base}, skipping.")

# 6. fastq -> fasta

In [ ]:
import os
import gzip
from Bio import SeqIO

# Input and output folder paths
input_folder = "fastq/stage2/1_assemble"
output_folder = "fastq/stage2/2_fastq_to_fasta"

# Create the output folder if it doesn't exist.
os.makedirs(output_folder, exist_ok=True)  

for filename in os.listdir(input_folder):
    # Process only files with .fastq or .fastq.gz extensions
    if filename.endswith(".fastq") or filename.endswith(".fastq.gz"):
        input_file = os.path.join(input_folder, filename)
        
        # Set output filename (.fasta extension)
        output_file = os.path.join(
            output_folder,
            filename.replace(".fastq.gz", ".fasta").replace(".fastq", ".fasta")
        )

        # Choose open mode based on gzip
        open_func = gzip.open if filename.endswith(".gz") else open

        # Read FASTQ and convert to FASTA
        with open_func(input_file, "rt") as fastq_file:
            # open in text mode
            records = list(SeqIO.parse(fastq_file, "fastq"))

        # Save as FASTA
        with open(output_file, "w") as fasta_file:
            SeqIO.write(records, fasta_file, "fasta")

        print(f"Converted: {filename} → {os.path.basename(output_file)}")

print("All conversions are done.")

# 7. Binary data reference seqeunce data generate

In [ ]:
from pathlib import Path

def generate_sequences_for_bit(bit_length: int):
    """
    Generate DNA sequences for all binary combinations of the given bit_length.
    (bit_length=8 -> 256 barcodes)
    """
    sequences = {}

    seq_0 = "ACTCATATACACACTTAATC"
    seq_1 = "ACTCATATACATACACTTAATC"
    prefix = "ACACTTAATC"

    for i in range(2 ** bit_length):
        binary_str = format(i, f'0{bit_length}b')
        sequence = ''.join(seq_1 if bit == '1' else seq_0 for bit in binary_str)
        full_sequence = prefix + sequence
        seq_id = f"seq_{i:04d}_{binary_str}"
        sequences[seq_id] = full_sequence

    return sequences

def write_fasta(sequences: dict, output_path: str):
    """Write sequences to a FASTA file."""
    with open(output_path, "w") as f:
        for seq_id, sequence in sequences.items():
            f.write(f">{seq_id}\n{sequence}\n")

# ===== Settings: exactly 8 bits (256 barcodes) =====
BIT_LENGTH = 8  # 2^8 = 256
output_dir = Path("reference_sequence")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "8bit_reference.fasta"
# ==================================================

seqs = generate_sequences_for_bit(BIT_LENGTH)
write_fasta(seqs, output_path)
print(f"✅ 8-bit (256) barcodes FASTA saved: {output_path}")

# 8. Reference sequence - Sample Matching

In [ ]:
# Index reference
!bwa index "reference_sequence/8bit_reference.fasta"

In [ ]:
%%bash
# Set the path to the reference sequence file
reference_file="reference_sequence/8bit_reference.fasta"

# Set the directory containing your filtered FASTA files
fasta_directory="fastq/stage2/2_fastq_to_fasta"
# Set the output directory for aligned SAM files
output_dir="fastq/stage2/3_align_sam"

# Make sure the output directory exists or create it if necessary
mkdir -p "$output_dir"

# Iterate through filtered FASTA files in the specified directory
for fasta_file in "$fasta_directory"/*_assemble.fasta; do
    # Generate an output file name based on the input filename
    output_file="$output_dir/$(basename "$fasta_file" .fasta).sam"

    # Perform the BWA alignment 
    bwa mem -M -t 4 "$reference_file" "$fasta_file" > "$output_file"

    echo "Alignment completed for $fasta_file. Result saved as $output_file"
done

## 8.1 sam to bam

In [ ]:
%%bash

# Set the path to the directory containing SAM files
sam_dir="fastq/stage2/3_align_sam"
# Set the output directory for BAM files
bam_dir="fastq/stage2/4_align_bam"

# Make sure the output directory exists or create it if necessary
mkdir -p "$bam_dir"

# Convert SAM files to BAM
for sam_file in "$sam_dir"/*.sam; do
    bam_file="$bam_dir/$(basename "$sam_file" .sam).bam"
    samtools view -bS "$sam_file" -o "$bam_file"
    echo "Conversion from $sam_file to $bam_file is complete."
done

## 8.2  Convert BAM to CSV

In [ ]:
import os
import pysam
import pandas as pd

# Input folder (path where BAM files are located)
input_folder = "fastq/stage2/4_align_bam"
# Output folder (path to save CSV files)
output_folder = "fastq/stage2/4_align_bam/csv"

# Create the output folder if it does not exist
os.makedirs(output_folder, exist_ok=True)

# Function to convert a BAM file to CSV, including optional fields
def bam_to_csv(bam_file, output_folder):
    output_csv = os.path.join(output_folder, os.path.basename(bam_file).replace(".bam", ".csv"))
    
    # Read the BAM file.
    with pysam.AlignmentFile(bam_file, "rb") as bam:
        records = []
        
        for read in bam:
            # Standard BAM fields.
            record = {
                "QNAME": read.query_name,
                "FLAG": read.flag,
                "RNAME": bam.get_reference_name(read.reference_id) if read.reference_id >= 0 else "*",
                "POS": read.reference_start + 1,
                "MAPQ": read.mapping_quality,
                "CIGAR": read.cigarstring if read.cigarstring else "*",
                "RNEXT": bam.get_reference_name(read.next_reference_id) if read.next_reference_id >= 0 else "*",
                "PNEXT": read.next_reference_start + 1 if read.next_reference_start >= 0 else 0,
                "TLEN": read.template_length,
                "SEQ": read.query_sequence if read.query_sequence else "*",
                "QUAL": read.qual if read.qual else "*",
            }
            
            # Add optional fields (tags).
            for tag, value in read.tags:
                record[tag] = value

            records.append(record)
    
    # Create a DataFrame from the list of records.
    df = pd.DataFrame(records)

    # Fill any missing optional fields with "*" instead of NaN for consistency.
    df = df.fillna("*")

    # Save the DataFrame to a CSV file.
    df.to_csv(output_csv, index=False)
    print(f"Converted: {os.path.basename(bam_file)} -> {os.path.basename(output_csv)}")
    return output_csv

# Find all BAM files in the input folder.
bam_files = [os.path.join(input_folder, f) for f in os.listdir(input_folder) if f.endswith(".bam")]

# Convert all found BAM files to CSV.
csv_files = []
for bam_file in bam_files:
    csv_file = bam_to_csv(bam_file, output_folder)
    csv_files.append(csv_file)

# Print the list of newly created CSV files.
csv_files

## 8.3 Filter Alignments by MAPQ Score

In [ ]:
import os
import pandas as pd
from pathlib import Path

# ===== Settings =====
input_dir = Path("fastq/stage2/4_align_bam/csv") # Input folder containing CSV files
output_dir = input_dir / "MAPQ_removed"  # Output folder for filtered CSV files
output_dir.mkdir(parents=True, exist_ok=True)

MAPQ_THRESHOLD = 10     # Keep rows where MAPQ > this value
KEEP_NAN = True         # Keep rows with NaN MAPQ values (e.g., unaligned reads)
# ====================

def process_one_csv(in_path: Path, out_dir: Path, mapq_threshold: int, keep_nan: bool = True):
    out_path = out_dir / in_path.name

    # Remove existing output file to avoid duplicates
    if out_path.exists():
        out_path.unlink()

    # Read input CSV
    try:
        df = pd.read_csv(in_path)
    except Exception as e:
        print(f"⚠️  Read fail: {in_path.name} -> {e}")
        return

    # Skip if MAPQ column does not exist
    if "MAPQ" not in df.columns:
        print(f"⚠️  Skip (no MAPQ column): {in_path.name}")
        return

    # Convert MAPQ column to numeric (invalid entries become NaN)
    m = pd.to_numeric(df["MAPQ"], errors="coerce")

    # Filtering mask: keep MAPQ > threshold, optionally keep NaN
    keep_mask = (m > mapq_threshold) | (m.isna() if keep_nan else False)

    kept = int(keep_mask.sum())
    removed = int((~keep_mask).sum())

    # Save filtered CSV
    df.loc[keep_mask].to_csv(out_path, index=False)
    print(
        f"✅ {in_path.name} → {out_path.name} | kept={kept}, removed={removed} "
        f"| threshold={mapq_threshold}, keep_nan={keep_nan}"
    )

def main():
    csv_files = sorted(input_dir.glob("*.csv"))
    if not csv_files:
        print(f"⚠️  No CSV files in {input_dir}")
        return

    for p in csv_files:
        process_one_csv(p, output_dir, MAPQ_THRESHOLD, KEEP_NAN)

if __name__ == "__main__":
    main()

# Histogram Data Analysis

## A. Generate Histogram Data from Aligned Reads(MAPQ filtered)

In [ ]:
import os
import pandas as pd

# Folder setup
input_folder = "fastq/stage2/4_align_bam/csv/MAPQ_removed"
histogram_folder = "fastq/stage2/5_histogram"
os.makedirs(histogram_folder, exist_ok=True)

# Process all CSV files in the input folder
files = [f for f in os.listdir(input_folder) if f.endswith('.csv')]

for file_name in files:
    file_path = os.path.join(input_folder, file_name)
    output_csv = os.path.join(histogram_folder, f"histogram_{file_name}")

    try:
        df = pd.read_csv(file_path, dtype=str)
        if 'RNAME' not in df.columns:
            print(f"Skipping file: {file_name} (no 'RNAME' column found)")
            continue

        # Count the occurrences of each unique RNAME
        rname_counts = df['RNAME'].value_counts().reset_index()
        rname_counts.columns = ['RNAME', 'Count']
        
        # Add metadata and calculate normalized counts
        rname_counts.insert(0, 'File_Name', file_name)
        rname_counts['Count'] = rname_counts['Count'].astype(int)
        total_count = rname_counts['Count'].sum()
        rname_counts['Normalized_Count'] = rname_counts['Count'] / total_count

        # Save the histogram data to a new CSV file
        rname_counts.to_csv(output_csv, index=False)
        print(f"✅ Saved full RNAME histogram: {output_csv}")

    except Exception as e:
        print(f"❌ Error processing file '{file_name}': {e}")

## C. Summarize Highlighted Read Counts into a CSV File

In [ ]:
import os
import re
import pandas as pd

# === Highlight Mapping (associates sample prefixes with their expected RNAME) ===
highlight_mapping = {
    "0N": "seq_013_00001101",
    "1D": "seq_035_00100011",
    "2S": "seq_082_01010010",
    "3SP": "seq_122_01111010",
    "4G": "seq_134_10000110",
    "5I": "seq_168_10101000",
    "6S": "seq_210_11010010",
    "7T": "seq_243_11110011"
}

# === Folders ===
histogram_folder = "fastq/stage2/5_histogram" 
summary_folder = "fastq/stage2/6_summary"
os.makedirs(summary_folder, exist_ok=True)
highlight_result_csv = os.path.join(summary_folder, "highlight_result.csv")


# === Helpers ===
def canonicalize_rname(x: str) -> str:
    """
    Normalize RNAME to a canonical form without zero-padding in the index part.
    Examples:
      'seq_0013_00001101' -> 'seq_13_00001101'
      'seq_013_00001101'  -> 'seq_13_00001101'
      'seq_13_00001101'   -> 'seq_13_00001101'
    If pattern doesn't match, return original stripped string.
    """
    s = str(x).strip()
    m = re.fullmatch(r"seq_(\d+)_([01]+)", s)
    if not m:
        return s
    idx = int(m.group(1))   # remove leading zeros
    bits = m.group(2)
    return f"seq_{idx}_{bits}"

# Pre-normalize mapping so it matches canonicalized RNAMEs in CSVs
normalized_mapping = {k: canonicalize_rname(v) for k, v in highlight_mapping.items()}

def extract_prefix_from_filename(file: str) -> str:
    """
    Robustly extract the sample prefix from filenames like:
      'histogram_DNA_Data_0N_assemble.csv' -> '0N'
      'histogram_DNA_Data_3SP26_assemble.csv' -> '3SP26'
    Strategy: take the token right before 'assemble'.
    """
    name = os.path.basename(file)
    if name.startswith("histogram_"):
        name = name[len("histogram_"):]
    if name.endswith(".csv"):
        name = name[:-4]
    tokens = name.split("_")
    # find token before 'assemble'
    try:
        i = tokens.index("assemble")
        if i - 1 >= 0:
            return tokens[i - 1]
    except ValueError:
        pass
    # fallback heuristic
    return tokens[2] if len(tokens) > 2 else (tokens[-1] if tokens else "")

# === Collect Highlight Summary Information ===
highlight_data = []
csv_files = [f for f in os.listdir(histogram_folder)
             if f.startswith("histogram_") and f.endswith(".csv")]

for file in csv_files:
    file_path = os.path.join(histogram_folder, file)
    try:
        df = pd.read_csv(file_path)

        # Ensure required columns exist
        if "RNAME" not in df.columns or "Count" not in df.columns:
            raise ValueError(f"Required columns 'RNAME' and 'Count' not found in {file}")

        # Canonicalize RNAMEs for reliable matching
        df["RNAME"] = df["RNAME"].map(canonicalize_rname)

        # Extract prefix robustly and get normalized highlight RNAME
        prefix = extract_prefix_from_filename(file)
        highlight_rname = normalized_mapping.get(prefix, "")

        # Ensure Count is integer
        df["Count"] = pd.to_numeric(df["Count"], errors="coerce").fillna(0).astype(int)
        total_count = int(df["Count"].sum())

        # Highlight stats
        highlight_count = int(df.loc[df["RNAME"] == highlight_rname, "Count"].sum()) if highlight_rname else 0
        highlight_percentage = (highlight_count / total_count * 100.0) if total_count > 0 else 0.0

        # Ratio vs second top
        sorted_counts = df["Count"].sort_values(ascending=False).to_list()
        second_max_count = sorted_counts[1] if len(sorted_counts) >= 2 else 0
        highlight_vs_second_ratio = (highlight_count / second_max_count) if second_max_count > 0 else 0.0

        # Keep the same 'File' field shape you used before (without the 'histogram_' prefix)
        file_name = file.replace("histogram_", "")

        highlight_data.append([
            file_name,
            highlight_count,
            total_count,
            highlight_percentage,
            highlight_rname,
            highlight_vs_second_ratio
        ])

    except Exception as e:
        print(f"❌ Error processing file '{file}': {e}")

# === Save the Summary to a CSV File (same columns/order as your original) ===
highlight_df = pd.DataFrame(highlight_data, columns=[
    'File',
    'Highlight_Count',
    'Total_Count',
    'Highlight_Percentage',
    'Highlight_RNAME',
    'Highlight_vs_SecondTop_Ratio'
])
highlight_df = highlight_df.sort_values(by='File')
highlight_df.to_csv(highlight_result_csv, index=False)

print(f"📌 Highlight summary saved to: {highlight_result_csv}")

## D. Plot Stacked Bar Graph top5_gray_rest_white_box

In [ ]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ---------------- Paths ----------------
histogram_folder = "fastq/stage2/5_histogram"
summary_folder = "fastq/stage2/6_summary"
os.makedirs(summary_folder, exist_ok=True)

# ---------------- Expected RNAME map (by sample prefix) ----------------
highlight_mapping = {
    "0N": "seq_013_00001101",
    "1D": "seq_035_00100011",
    "2S": "seq_082_01010010",
    "3SP": "seq_122_01111010",
    "4G": "seq_134_10000110",
    "5I": "seq_168_10101000",
    "6S": "seq_210_11010010",
    "7T": "seq_243_11110011"
}

# ---------------- Visual options ----------------
# If True, use only the prefix (e.g., 3SP, 3SP) as x-axis label; else use full sample name.
USE_PREFIX_LABELS = True

# Rank-based colors: 1st=red, 2nd~4th gray shades (RGB 0–1), 5+=white merged
RANK_COLORS = [
    "red",              # 1st
    (0.30, 0.30, 0.30), # 2nd
    (0.5, 0.5, 0.5), # 3rd
    (0.7, 0.7, 0.7), # 4th
]

# ---------------- Load per-sample data ----------------
sample_rname_dfs = {}
for file_name in sorted(os.listdir(histogram_folder)):
    if not (file_name.lower().startswith("histogram_") and file_name.endswith(".csv")):
        continue

    file_path = os.path.join(histogram_folder, file_name)
    sample_name = file_name.replace("histogram_", "").replace(".csv", "")  # e.g., DNA_Data_3SP_assemble

    # robust prefix extraction: token right before "_assemble"
    m = re.search(r"([A-Za-z0-9]+)(?=_assemble$)", sample_name)
    prefix = m.group(1) if m else None

    try:
        df = pd.read_csv(file_path)
    except Exception as e:
        print(f"[ERROR] Failed to read {file_name}: {e}")
        continue

    if "RNAME" not in df.columns or "Count" not in df.columns:
        print(f"[SKIP] {file_name}: missing RNAME/Count columns.")
        continue

    # Normalize safely (avoid zero division)
    df["Count"] = df["Count"].astype(int)
    total = df["Count"].sum()
    if total == 0:
        print(f"[SKIP] {file_name}: total Count is zero.")
        continue

    df["Normalized_Count"] = df["Count"] / total
    df = df.sort_values(by="Count", ascending=False).reset_index(drop=True)

    sample_rname_dfs[sample_name] = (prefix, df)

if not sample_rname_dfs:
    print("[WARN] No valid histogram CSV files found. Check folder and filenames.")
else:
    print(f"[INFO] Loaded {len(sample_rname_dfs)} samples.")

# ---------------- Plot ----------------
fig, ax = plt.subplots(figsize=(48, 12))

for sample_name, (prefix, df) in sample_rname_dfs.items():
    # expected highlight RNAME by prefix (may be None if prefix not in map)
    highlight_rname = highlight_mapping.get(prefix)

    # split top4 and the rest
    top4 = df.head(4).copy()
    rest_sum = df["Normalized_Count"].iloc[4:].sum() if len(df) > 4 else 0.0

    # x label
    x_label = prefix if (USE_PREFIX_LABELS and prefix) else sample_name

    bottom = 0.0
    for i, row in top4.iterrows():
        height = float(row["Normalized_Count"])
        rname = str(row["RNAME"]).strip()

        # 1st should be red if it equals expected highlight; otherwise still rank color
        if highlight_rname and rname == highlight_rname:
            bar_color = RANK_COLORS[0]
        else:
            # i: 0..3  →  choose rank color slot
            bar_color = RANK_COLORS[i if i < len(RANK_COLORS) else -1]

        ax.bar(
            x_label,
            height,
            bottom=bottom,
            color=bar_color,
            edgecolor="black",
            linewidth=0.2,
        )
        bottom += height

    if rest_sum > 0:
        ax.bar(
            x_label,
            rest_sum,
            bottom=bottom,
            color="white",
            edgecolor="black",
            linewidth=0.2,
        )

# reference line and styling
ax.axhline(y=0.5, color="gray", linestyle="--", linewidth=1)
ax.set_ylabel("Normalized Count", fontsize=20)
ax.set_xlabel("Sample", fontsize=20)
ax.set_title(
    "Stacked Bar Chart (1st=Red, 2nd-4th=Gray Shades, Other=White)",
    fontsize=18,
)
ax.tick_params(axis="x", labelsize=18)
ax.tick_params(axis="y", labelsize=18)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

# ---------------- Save ----------------
png_path = os.path.join(summary_folder, "stacked_bar_rank_color_RGB.png")
svg_path = os.path.join(summary_folder, "stacked_bar_rank_color_RGB.svg")
plt.savefig(png_path, dpi=300)
plt.savefig(svg_path)
# pdf_path = os.path.join(summary_folder, "stacked_bar_rank_color_RGB.pdf"); plt.savefig(pdf_path)
plt.close()

print("✅ Saved outputs:")
print(" -", png_path)
print(" -", svg_path)
# print(" -", pdf_path)

# Error Analysis

In [ ]:
import pandas as pd
import os
import re
import numpy as np

# --- 1. Set Answer Key ---
answer_data = {
    "0N": "seq_013_00001101",
    "1D": "seq_035_00100011",
    "2S": "seq_082_01010010",
    "3SP": "seq_122_01111010",
    "4G": "seq_134_10000110",
    "5I": "seq_168_10101000",
    "6S": "seq_210_11010010",
    "7T": "seq_243_11110011"
}
answer_key_map = {key: re.search(r'([01]{8})$', value).group(1) for key, value in answer_data.items()}

# --- 2. File Processing ---
input_folder = "fastq/stage2/5_histogram"
output_folder = "fastq/stage2/6_summary"
output_path = os.path.join(output_folder, "summary_combined_v2.csv")

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

try:
    files = sorted([f for f in os.listdir(input_folder) if f.endswith('.csv') and f.startswith("histogram_")])
except FileNotFoundError:
    print(f"❌ Error: Folder '{input_folder}' not found.")
    files = []

rows_list = []

for file_name in files:
    try:
        df = pd.read_csv(os.path.join(input_folder, file_name))
        total_count = df['Count'].sum()
        
        # 포지션별 카운트 초기화
        pos_counts = [{'0': 0, '1': 0} for _ in range(8)]
        for _, row in df.iterrows():
            match = re.search(r'seq_[^_]+_([01]{8})', str(row.get('RNAME', '')))
            if match:
                bits = match.group(1)
                for i, bit in enumerate(bits):
                    pos_counts[i][bit] += int(row['Count'])
        
        # 정답지 매칭
        answer_key = next((answer_key_map[k] for k in answer_key_map if k in file_name), None)
        
        # --- 행 데이터 생성 (3개 층) ---
        zero_row = {"File_Name": file_name, "Total_Count": total_count, "Type": "Zeros_Count"}
        one_row = {"File_Name": "", "Total_Count": "", "Type": "Ones_Count"}
        acc_row = {"File_Name": "", "Total_Count": "", "Type": "Accuracy"}
        
        accuracies = []
        for i in range(8):
            z, o = pos_counts[i]['0'], pos_counts[i]['1']
            zero_row[i+1] = z
            one_row[i+1] = o
            
            acc = 0
            if answer_key:
                correct_bit = answer_key[i]
                acc = (pos_counts[i][correct_bit] / (z + o)) if (z + o) > 0 else 0
            acc_row[i+1] = acc
            accuracies.append(acc)
            
        acc_row["Avg_Accuracy"] = np.mean(accuracies)
        
        # 리스트에 추가
        rows_list.extend([zero_row, one_row, acc_row])
        print(f"✅ {file_name} processed.")

    except Exception as e:
        print(f"❌ Error in {file_name}: {e}")

# --- 3. Save and Final Summary ---
if rows_list:
    final_df = pd.DataFrame(rows_list)
    
    # 컬럼 순서 정리
    cols = ["File_Name", "Total_Count", "Type"] + [i for i in range(1, 9)] + ["Avg_Accuracy"]
    final_df = final_df[cols]
    
    # 전체 평균 행 추가 (모든 Accuracy 행의 평균)
    overall_acc = final_df[final_df["Type"] == "Accuracy"]["Avg_Accuracy"].mean()
    summary_footer = pd.DataFrame([{"File_Name": "Total Avg", "Avg_Accuracy": overall_acc}])
    final_df = pd.concat([final_df, summary_footer], ignore_index=True)

    final_df.to_csv(output_path, index=False)
    print(f"\n📂 Saved to: {output_path}")